# SOXX趋势 × SOXL相对路径落后：三资产联合边际的必要秩检验

两个旧输入的乘积和标签在评价年份前冻结；只读Qlib原始特征与成熟H5价格标签，不拟合、不跑账户。

In [1]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
import hashlib,json
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R
from scipy.stats import spearmanr

root=Path.cwd();card=root/"research/stage1_soxx_trend_decay_joint_margin_rank_20261024"
plan_path=card/"plan.json";plan=json.loads(plan_path.read_text());src=plan["sources"]
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan["frozen"] and plan["budget"]["new_market_GETs"]==plan["budget"]["new_model_fits"]==plan["budget"]["new_native_accounts"]==0
assert sha(root/src["interval_result"])==src["interval_result_sha256"]
assert sha(root/src["closed_broad_rebound_summary"])==src["closed_broad_rebound_summary_sha256"]
a=root/src["features_card"];b=root/src["labels_card"]
assert sha(a/"summary.json")==src["features_summary_sha256"] and sha(b/"summary.json")==src["labels_summary_sha256"]
sa=json.loads((a/"summary.json").read_text());sb=json.loads((b/"summary.json").read_text())
assert sa["nas_evidence"]["sha256"]==src["features_NAS_sha256"] and sa["nas_evidence"]["size_bytes"]==src["features_NAS_size_bytes"]
assert sb["nas_evidence"]["sha256"]==src["labels_NAS_sha256"] and sb["nas_evidence"]["size_bytes"]==src["labels_NAS_size_bytes"]
assert sb["nas_readback"]["verified"] and sb["nas_readback"]["sha256"]==src["labels_NAS_sha256"]
aa=a/"qlib_recorder"/src["features_experiment_id"]/src["features_recorder_id"]/"artifacts"
ba=b/"qlib_recorder"/src["labels_experiment_id"]/src["labels_recorder_id"]/"artifacts"
assert sha(aa/"inputs/features")==src["features_artifact_sha256"] and sha(aa/"inputs/timing")==src["timing_artifact_sha256"]
assert sha(ba/"inputs/dataset")==src["labels_dataset_sha256"]
assert sha(root/src["provider"]/"qlib_store_manifest.json")==src["provider_manifest_sha256"]
qlib.init(provider_uri=str((root/src["provider"]).resolve()),region="us",expression_cache=None,dataset_cache=None,kernels=1)
calendar=pd.DatetimeIndex(D.calendar(start_time="2021-01-01",end_time="2022-12-30",freq="day"))
assert len(calendar)==503
with R.uri_context(str(a/"qlib_recorder")):
 rec=R.get_recorder(experiment_id=src["features_experiment_id"],recorder_id=src["features_recorder_id"])
 features=rec.load_object("inputs/features")
 timing=rec.load_object("inputs/timing")
with R.uri_context(str(b/"qlib_recorder")):
 rec=R.get_recorder(experiment_id=src["labels_experiment_id"],recorder_id=src["labels_recorder_id"])
 dataset=rec.load_object("inputs/dataset")
train_label=dataset.prepare("train",col_set=["label","meta"])
assert len(features)==len(timing)==3966 and len(train_label)==3020
assert features.index.equals(timing.index)
decision=pd.to_datetime(timing.reindex(calendar).decision_at,utc=True).dt.tz_convert("America/New_York").dt.date
assert all(d>x.date() for d,x in zip(decision,calendar))
assert set(["soxx_ret60","levered_decay_20"])<=set(features.columns)
scaler=features.loc[plan["evaluation"]["scaler_train_signal_dates"][0]:plan["evaluation"]["scaler_train_signal_dates"][1],["soxx_ret60","levered_decay_20"]]
med=scaler.median();sd=scaler.std(ddof=0)
assert len(scaler)>2000 and np.isfinite(med.to_numpy()).all() and (sd>0).all()
f=features.reindex(calendar).loc[:,["soxx_ret60","levered_decay_20"]]
assert f.notna().all().all()
z_trend=(f.soxx_ret60-med.soxx_ret60)/sd.soxx_ret60
z_decay=(f.levered_decay_20-med.levered_decay_20)/sd.levered_decay_20
score=z_trend.clip(lower=0)*(-z_decay).clip(lower=0)
assert np.isfinite(score.to_numpy()).all()
label=train_label.loc[pd.DatetimeIndex(train_label.index.get_level_values("datetime")).isin(calendar)].copy()
label.index=pd.DatetimeIndex(label.index.get_level_values("datetime"))
assert label.index.equals(calendar)
cash=label[("label","cash_advantage_H5")].astype(float)
relative=label[("label","SOXL_relative_H5")].astype(float)
mature=label[("meta","train_mature")].astype(bool)
y=pd.concat([-cash,-relative],axis=1).min(axis=1)
mask=mature & cash.notna() & relative.notna() & score.notna()
assert int(mask.sum())>=plan["evaluation"]["minimum_mature_scored"]
whole=float(spearmanr(score.loc[mask],y.loc[mask]).statistic)
annual={}
for year in (2021,2022):
 pick=mask&(calendar.year==year)
 nonzero=int((score.loc[pick]>0).sum())
 annual[str(year)]={"rows":int(pick.sum()),"nonzero_factor_rows":nonzero,"Spearman":float(spearmanr(score.loc[pick],y.loc[pick]).statistic)}
support_gate=all(a["nonzero_factor_rows"]>=plan["evaluation"]["minimum_nonzero_factor_per_year"] for a in annual.values())
fixed=plan["evaluation"]["bootstrap"]
n=len(calendar)
if support_gate:
 fixed=plan["evaluation"]["bootstrap"]
 rng=np.random.default_rng(fixed["seed"]);n=len(calendar);jump=1/fixed["mean_block_sessions"]
 fx=score.to_numpy(float);fy=y.where(mask).to_numpy(float)
 rep=[]
 for _ in range(fixed["replicates"]):
  idx=np.empty(n,dtype=int);idx[0]=rng.integers(n)
  for i in range(1,n):idx[i]=rng.integers(n) if rng.random()<jump else (idx[i-1]+1)%n
  xx,yy=fx[idx],fy[idx];valid=np.isfinite(xx)&np.isfinite(yy)
  if valid.sum()>=100:
   z=float(spearmanr(xx[valid],yy[valid]).statistic)
   if np.isfinite(z):rep.append(z)
 assert len(rep)>=fixed["minimum_valid_replicates"]
 lower=float(np.quantile(rep,fixed["one_sided_alpha"]))
else:
 rep=[]
 lower=None
gates={"nonzero_support":support_gate,"full_positive":whole>0,"both_years_positive":all(a["Spearman"]>0 for a in annual.values()),"block_lower_positive":lower is not None and lower>0}
result={"status":"necessary_trend_decay_joint_margin_rank_pass" if all(gates.values()) else "not_promoted_fixed_trend_decay_joint_margin_rank_failed",
        "plan_sha256":sha(plan_path),"features_recorder_id":src["features_recorder_id"],"labels_recorder_id":src["labels_recorder_id"],
        "training_scaler_rows":len(scaler),"full_eval_calendar_rows":n,"mature_scored_rows":int(mask.sum()),
        "factor_train_median":{k:float(v) for k,v in med.items()},"factor_train_SD":{k:float(v) for k,v in sd.items()},
        "full_Spearman":whole,"annual":annual,"bootstrap":{"valid_replicates":len(rep),"mean_block_sessions":fixed["mean_block_sessions"],"one_sided_alpha":fixed["one_sided_alpha"],"lower_bound":lower},
        "gates":gates,"new_market_GETs":0,"new_model_fits":0,"new_native_accounts":0,"new_bootstraps":1 if support_gate else 0,
        "independent_validation":False,"SOXX_goal_achieved":False,
        "limits":["The joint H5 price margin is a necessary gross information label and not a Tiered fee-net funded account",
                  "The two inputs already belong to original39; interaction novelty and incremental information are not established by a rank pass",
                  "All evaluation years and price history are reused development feedback; no forward evidence"]}
(card/"result.json").write_text(json.dumps(result,ensure_ascii=False,indent=2)+"\n")
print({"status":result["status"],"full":whole,"annual":annual,"block_lower":lower,"gates":gates})


{'status': 'not_promoted_fixed_trend_decay_joint_margin_rank_failed', 'full': 0.07497034755750784, 'annual': {'2021': {'rows': 252, 'nonzero_factor_rows': 111, 'Spearman': 0.06682824454492528}, '2022': {'rows': 245, 'nonzero_factor_rows': 21, 'Spearman': -0.06284393373343455}}, 'block_lower': None, 'gates': {'nonzero_support': False, 'full_positive': True, 'both_years_positive': False, 'block_lower_positive': False}}


## 中文结论

从原39项输入中预定两个已知因子：SOXX前60会话收益与SOXL前20会话相对三倍SOXX路径的观察差额。两者仅用2011–2020训练期中心与标准差归一化，固定非负趋势乘以非负相对落后。标签是下一H5开盘区间里SOXX同时胜过现金与SOXL的**较小毛价格优势**；这与旧单独现金、SOXL相对H5接收器不同，但仍不是费用后完整账户。

Qlib完整2021–2022评价日历503日，497行双标签成熟。全体Spearman点值 **+0.07497**；2021为 **+0.06683**、非零因子111行；2022为 **−0.06284**、非零仅 **21行**，不足冻结的每年50行支持门。两个完整年符号也不一致。按预定停止条件，未运行区块bootstrap、增量拟合或资金账户。2023事后强趋势现金拖累不能替代2022缺少训练／评价支持，更不能成为交易阈值。

首次notebook在支持门已失败后，仅因结果序列化变量未在跳过bootstrap分支初始化而报错；保存原计划与`attempt1_result_record_repair.json`，第二次只补全结果，不改公式、门槛或标签。决定 `not_promoted_fixed_trend_decay_joint_margin_rank_failed`。不降低非零门、挑2021、翻2022方向或改ReLU阈值救援。原Stage1完整0bp、2/5bp及SOXX v2门均未通过。
